In [1]:
import json
import sys
import re
import pandas as pd
from rich import print as rp
from collections import Counter
from pathlib import Path
from datetime import datetime

from splink import Linker, DuckDBAPI, SettingsCreator, block_on
import splink.comparison_library as cl
import splink.comparison_level_library as cll
from splink.blocking_analysis import count_comparisons_from_blocking_rule


nb_dir = Path.cwd()
project_root = nb_dir.parent.parent
sys.path.insert(0, str(project_root))

In [2]:
people_file = Path(project_root / "data_reload/db_exports/people-2026-08-13.json")

people_df = pd.DataFrame((json.load(open(people_file))))
# people_df["unique_id"] = people_df.index ---> NO!!! This created a FAKE person_id!
people_df["unique_id"] = people_df["person_id"]

rp(len(people_df))

8936

In [3]:
db_api = DuckDBAPI()

settings = SettingsCreator(
    link_type="dedupe_only",
    blocking_rules_to_generate_predictions=[
        block_on("family_name"),
        block_on("family_name", "substr(given_names, 1, 1)"),
    ],
    comparisons=[
        cl.CustomComparison(
            output_column_name="family_name",
            comparison_levels=[
                cll.NullLevel("family_name"),
                cll.ExactMatchLevel("family_name").configure(
                    tf_adjustment_column="family_name",
                    m_probability=0.9,        # ← if two records are the same person,
                    fix_m_probability=True,   #   ~90% of the time the surname is written
                ),                            #   exactly the same. Locked so EM can't undo it.
                cll.JaroWinklerLevel("family_name", 0.92),
                cll.JaroWinklerLevel("family_name", 0.88),
                cll.JaroWinklerLevel("family_name", 0.7),
                cll.ElseLevel(),
            ],
        ),
        cl.NameComparison("given_names"),
    ],
    retain_intermediate_calculation_columns=True
)

# br = block_on("substr(given_names, 1, 1)", "family_name")

# count_comparisons_from_blocking_rule(
#     table_or_tables=people_df,
#     blocking_rule=br,
#     link_type="dedupe_only",
#     db_api=db_api
# )


linker = Linker([people_df], settings, db_api=db_api)

linker.training.estimate_probability_two_random_records_match([block_on("family_name")], recall=0.7)
linker.training.estimate_u_using_random_sampling(max_pairs=1e8)
linker.training.estimate_parameters_using_expectation_maximisation(block_on("family_name"))
linker.training.estimate_parameters_using_expectation_maximisation(block_on("given_names"))

# linker.misc.save_model_to_json("splink_model.json", overwrite=True)


Probability two random records match is estimated to be  0.000179.
This means that amongst all possible pairwise record comparisons, one in 5,595.74 are expected to match.  With 39,921,580 total possible comparisons, we expect a total of around 7,134.29 matching pairs
----- Estimating u probabilities using random sampling -----


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - family_name (some m values are not trained).
    - given_names (no m values are trained).

----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."family_name" = r."family_name"

Parameter estimates will be made for the following comparison(s):
    - given_names

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - family_name

Iteration 1: Largest change in params was -0.828 in the m_probability of given_names, level `Exact match on given_names`
Iteration 2: Largest change in params was 0.0921 in the m_probability of given_names, level `All other comparisons`
Iteration 3: Largest change in params was 0.0409 in the m_probability of given_names, level `All other comparisons`
Iteration 4: Largest change in params was -0.023 in the m_probability of given_names, le

<EMTrainingSession, blocking on l."given_names" = r."given_names", deactivating comparisons given_names>

In [6]:

predictions = linker.inference.predict()


edges = predictions.as_pandas_dataframe()

edges = edges.sort_values("match_probability", ascending=False)
# records = edges.head(20).to_dict(orient="records")
# linker.visualisations.waterfall_chart(records)
cols = edges[["match_weight", "match_probability", "family_name_l", "family_name_r", "given_names_l", "given_names_r", "unique_id_l", "unique_id_r"]]

review = cols.copy()

review = review.rename(columns={
    "family_name_l": "family_name_1", "family_name_r": "family_name_2",
    "given_names_l": "given_names_1", "given_names_r": "given_names_2",
    "unique_id_l": "unique_id_1", "unique_id_r": "unique_id_2",
})

review["decision"] = ""
# review.to_csv("deduplication_review.csv", index=False)
review.to_excel("deduplication_review.xlsx", engine="openpyxl")

# rp(records)


Blocking time: 0.01 seconds
Predict time: 0.04 seconds
